# N5 — Capstone: Navigate the Real Scene

### An interactive bird's-eye-view (BEV) autonomy simulator

Across N1–N4 we built the pieces of a self-driving stack in isolation. This
capstone fuses them into **one closed-loop simulation** you can poke at: an ego
vehicle drives a **real KITTI trajectory** through a BEV scene with cars on the
road, perceives them, and either **steers around them** — or clips one. Every knob
on the dashboard ties back to a notebook:

| Earlier notebook | Its role in this capstone |
|---|---|
| **N1** — Camera–LiDAR Projection | Defines the **metric BEV ground-plane frame** everything lives in; the optional bonus cell back-projects real detections into it |
| **N2** — Kalman From Scratch | The **constant-velocity Kalman filter** behind every tracked car; **coasting** when a sensor drops out |
| **N3** — LiDAR 3D Tracking | **Per-object tracking** — Hungarian association with IDs and a tentative→confirmed→coasting lifecycle, lifted into the BEV plane; also the **source of realistic obstacle sizes** |
| **N4** — Path Tracking & Control | The **bicycle model + path-tracking controller** (pure-pursuit / Stanley) that steers the ego along the route |
| **+ new** | A **potential-field avoidance** layer that steers the ego *around* the obstacles, backed by an emergency brake |

**The payoff:** at the end you get a dashboard where you tune the controller and the
avoidance, lay out the obstacle course, and inject sensor noise — and watch, in BEV,
whether the ego **threads the course or clips a car**.

## 1. Setup

In [ ]:
%pip install numpy scipy matplotlib ipywidgets pykitti --quiet

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.animation import FuncAnimation
from matplotlib.transforms import Affine2D
from IPython.display import HTML, display
from scipy.optimize import linear_sum_assignment
from dataclasses import dataclass
import os, glob

np.random.seed(0)
print("Libraries loaded.")

## 2. The shared world: a BEV frame anchored on real KITTI data

Everything in this notebook lives in a single **bird's-eye-view metric frame** —
X points forward, Y points left, units are meters. This is the same ground plane
N1 back-projects into and the same East-North convention N3 used for KITTI's
GPS/IMU (OxTS) data.

The **ego vehicle's reference path is the real trajectory** the KITTI car drove
on drive `2011_09_26_0005` — parsed from OxTS latitude/longitude exactly as in
N3. If KITTI isn't available we fall back to a representative synthetic course
(straight → sweeping curve → straight) so the notebook always runs.

---

### From GPS coordinates to a local metric frame

KITTI's OxTS unit logs **latitude $\phi$ and longitude $\lambda$**, not meters. We convert each fix to a local **East–North** frame centered on the first sample $(\phi_0, \lambda_0)$ with the equirectangular (flat-earth) approximation:

$$x_E = (\lambda - \lambda_0)\,\cos\phi_0 \cdot R_{\deg}, \qquad y_N = (\phi - \phi_0)\cdot R_{\deg}$$

where $R_{\deg} \approx 111{,}320$ m is the length of one degree of latitude, and the $\cos\phi_0$ factor accounts for longitude lines bunching together away from the equator. This is accurate to centimeters over the few-kilometer span of one drive — the same conversion N3 used for its KITTI example.

We then **re-sample the trajectory to a uniform arclength spacing** $\Delta s \approx 0.5$ m (`densify_path`), so lookahead and Frenet lookups behave consistently no matter how fast the car was driving when each sample was logged.

In [ ]:
def normalize_angle(a):
    """Wrap angle to [-pi, pi]."""
    return (a + np.pi) % (2 * np.pi) - np.pi


def load_kitti_ego_path(base="kitti_data", date="2011_09_26", drive="0005", ds=0.5):
    """Real ego path from KITTI OxTS (lat/lon -> local East-North meters).
    Returns a densified (N,2) array of [x_forward, y_left] waypoints, or None."""
    oxts_dir = os.path.join(base, date, f"{date}_drive_{drive}_sync", "oxts", "data")
    files = sorted(glob.glob(os.path.join(oxts_dir, "*.txt")))
    if not files:
        return None
    oxts = np.array([np.loadtxt(f) for f in files])
    lat0 = np.radians(oxts[0, 0])
    east = (oxts[:, 1] - oxts[0, 1]) * np.cos(lat0) * 111320.0
    north = (oxts[:, 0] - oxts[0, 0]) * 111320.0
    raw = np.column_stack([east, north])          # x=east(forward-ish), y=north(left-ish)
    return densify_path(raw, ds)


def synthetic_path(ds=0.5):
    """Fallback: straight (60 m) -> gentle left curve -> straight (60 m)."""
    segs = []
    n1 = int(60 / ds)
    segs.append(np.column_stack([np.linspace(0, 60, n1), np.zeros(n1)]))
    R, arc = 80.0, 40.0
    th = np.linspace(0, arc / R, int(arc / ds))
    cx, cy = 60.0, R
    sx, sy = cx + R * np.sin(th), cy - R * np.cos(th)
    segs.append(np.column_stack([sx, sy]))
    ex, ey, hd = sx[-1], sy[-1], arc / R
    n3 = int(60 / ds)
    segs.append(np.column_stack([ex + np.linspace(0, 60 * np.cos(hd), n3),
                                 ey + np.linspace(0, 60 * np.sin(hd), n3)]))
    return np.vstack(segs)


def densify_path(raw, ds=0.5):
    """Resample a coarse polyline to ~uniform ds spacing."""
    seg = np.hypot(np.diff(raw[:, 0]), np.diff(raw[:, 1]))
    s = np.concatenate([[0], np.cumsum(seg)])
    n = max(int(s[-1] / ds), 2)
    su = np.linspace(0, s[-1], n)
    return np.column_stack([np.interp(su, s, raw[:, 0]), np.interp(su, s, raw[:, 1])])


# Try real KITTI; fall back to synthetic
PATH = load_kitti_ego_path()
if PATH is None:
    print("KITTI OxTS not found - using synthetic reference path.")
    PATH = synthetic_path()
else:
    print(f"Loaded real KITTI ego path: {len(PATH)} waypoints.")

print(f"Path extent: X [{PATH[:,0].min():.0f}, {PATH[:,0].max():.0f}] m, "
      f"Y [{PATH[:,1].min():.0f}, {PATH[:,1].max():.0f}] m")

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(PATH[:, 0], PATH[:, 1], "-", color="gray", lw=2)
ax.plot(PATH[0, 0], PATH[0, 1], "go", ms=10, label="start")
ax.set_aspect("equal"); ax.grid(alpha=0.3)
ax.set_xlabel("X — forward (m)"); ax.set_ylabel("Y — left (m)")
ax.set_title("Ego reference path (BEV)"); ax.legend()
plt.show()

## 3. Placing obstacles on the route: the Frenet frame

To drop cars *on the road the ego is driving* — at controlled distances and lateral
offsets, even as the road curves — we use **path-relative (Frenet) coordinates**:
an arclength `s` along the reference path and a lateral offset `d` (positive = left
of the path). A point on the path is `d = 0`; a car nudged into the lane is a small
`d`. This is exactly how `make_obstacle_course` lays out the obstacles.

---

### The Frenet transform

Let the reference path be a curve $\mathbf{r}(s)$ parameterized by arclength $s$. At each point it has a unit **tangent** and a unit **left normal**:

$$\hat{\mathbf{t}}(s) = \begin{bmatrix}\cos\psi(s)\\ \sin\psi(s)\end{bmatrix}, \qquad \hat{\mathbf{n}}(s) = \begin{bmatrix}-\sin\psi(s)\\ \cos\psi(s)\end{bmatrix}$$

where $\psi(s)$ is the local path heading. A Frenet coordinate $(s, d)$ maps to the world by walking $s$ along the path and stepping $d$ along the left normal:

$$\mathbf{p}(s, d) = \mathbf{r}(s) + d\,\hat{\mathbf{n}}(s)$$

`path_frame` returns the origin and heading at a given arclength; we offset by $d$ along $[-\sin\psi,\ \cos\psi]$ to place each obstacle, so a car at constant $d$ sits **parallel to the road**, curving along with it.

In [ ]:
def path_arclength(path):
    seg = np.hypot(np.diff(path[:, 0]), np.diff(path[:, 1]))
    return np.concatenate([[0.0], np.cumsum(seg)])


def path_frame(path, idx):
    """(origin xy, tangent heading) at a path index."""
    i = int(np.clip(idx, 0, len(path) - 2))
    d = path[i + 1] - path[i]
    return path[i], np.arctan2(d[1], d[0])


def frenet_to_world(path, s_arr, s, d):
    """(arclength s, lateral offset d) -> world [x, y, heading]."""
    i = int(np.clip(np.searchsorted(s_arr, s) - 1, 0, len(path) - 2))
    seg = path[i + 1] - path[i]
    h = np.arctan2(seg[1], seg[0])
    along = (s - s_arr[i]) / max(s_arr[i + 1] - s_arr[i], 1e-9)
    base = path[i] + along * seg
    nrm = np.array([-np.sin(h), np.cos(h)])   # left normal
    p = base + d * nrm
    return p[0], p[1], h

print("Frenet helpers ready.")

## 4. The ego vehicle — bicycle model + path tracking (N4)

The ego is driven by N4's **kinematic bicycle model** and a **pure-pursuit** (or
**Stanley**) controller that steers toward the reference path. Nothing new here —
we are reusing N4 directly. The controller is the first knob participants will
tune (lookahead distance, controller type, target speed).

---

### Recap of the equations (from N4)

**Kinematic bicycle model** — state $[x, y, \psi, v]$, controls steering $\delta$ and acceleration $a$, wheelbase $L$:

$$\dot{x} = v\cos\psi, \quad \dot{y} = v\sin\psi, \quad \dot{\psi} = \frac{v}{L}\tan\delta, \quad \dot{v} = a$$

stepped forward by $\Delta t$ in `bicycle_step`.

**Pure pursuit** — aim at a lookahead point $(x_L, y_L)$ a distance $L_d$ ahead; with $\alpha$ the angle from the heading to that point, the steering that fits a circular arc through the rear axle and the point is:

$$\alpha = \operatorname{atan2}(y_L - y,\; x_L - x) - \psi, \qquad \delta = \arctan\!\left(\frac{2L\sin\alpha}{L_d}\right)$$

**Stanley** — cancel heading error $\psi_{\text{path}}-\psi$ and cross-track error $e$ (gain $k$, speed $v$):

$$\delta = (\psi_{\text{path}} - \psi) + \arctan\!\left(\frac{k\,e}{v}\right)$$

The lookahead $L_d$ (pure pursuit) and gain $k$ (Stanley) are the two steering knobs on the dashboard. See N4 for the full derivation and tuning sweep.

In [ ]:
L_WB = 2.5                      # wheelbase (m)
MAX_STEER = np.radians(35)

def bicycle_step(state, delta, a, dt, L=L_WB):
    """One step of the kinematic bicycle model. state = [x, y, psi, v]."""
    x, y, psi, v = state
    x += v * np.cos(psi) * dt
    y += v * np.sin(psi) * dt
    psi = normalize_angle(psi + v / L * np.tan(delta) * dt)
    v = max(v + a * dt, 0.0)
    return np.array([x, y, psi, v])


def find_lookahead_point(state, path, Ld):
    x, y = state[0], state[1]
    d = np.hypot(path[:, 0] - x, path[:, 1] - y)
    nearest = int(np.argmin(d))
    for i in range(nearest, len(path)):
        if np.hypot(path[i, 0] - x, path[i, 1] - y) >= Ld:
            return path[i], nearest
    return path[-1], nearest


def pure_pursuit_steering(state, la_pt, L=L_WB):
    x, y, psi, v = state
    alpha = normalize_angle(np.arctan2(la_pt[1] - y, la_pt[0] - x) - psi)
    Ld = max(np.hypot(la_pt[0] - x, la_pt[1] - y), 0.1)
    return np.clip(np.arctan2(2 * L * np.sin(alpha), Ld), -MAX_STEER, MAX_STEER)


def stanley_steering(state, path, k, L=L_WB):
    x, y, psi, v = state
    fx, fy = x + L * np.cos(psi), y + L * np.sin(psi)
    d = np.hypot(path[:, 0] - fx, path[:, 1] - fy)
    i = int(np.argmin(d)); j = min(i + 1, len(path) - 1)
    pdx, pdy = path[j] - path[max(i - 1, 0)]
    he = normalize_angle(np.arctan2(pdy, pdx) - psi)
    cte = (pdx * (fy - path[i, 1]) - pdy * (fx - path[i, 0])) / (np.hypot(pdx, pdy) + 1e-9)
    return np.clip(he + np.arctan2(k * (-cte), max(abs(v), 0.5)), -MAX_STEER, MAX_STEER)


def cross_track_error(state, path):
    x, y = state[0], state[1]
    d = np.hypot(path[:, 0] - x, path[:, 1] - y)
    i = int(np.argmin(d)); j = min(i + 1, len(path) - 1)
    pdx, pdy = path[j] - path[max(i - 1, 0)]
    return (pdx * (y - path[i, 1]) - pdy * (x - path[i, 0])) / (np.hypot(pdx, pdy) + 1e-9)

print("Ego model + controllers ready (from N4).")

## 5. The obstacles — real-derived static cars on the route

N3 detects other vehicles on the real KITTI scene, but they sit mostly in adjacent
and oncoming lanes — not blocking the ego's own path (that's exactly why the drive
was recorded). So instead of replaying them literally, we **sample realistic cars**
and **place them on the route**.

`make_obstacle_course` drops a handful of **static** cars at arclength fractions
along the path, jittered laterally (in Frenet `d`) so the ego must actually steer
around them. Their size comes straight from N3's clustering gates — a car ≈
**4.2 m long × 1.8 m wide × 1.5 m tall** (`CAR_DIMS`). Perception-grounded sizes,
deliberately **arranged** into a course: the "simulate the other stuff" half of a
real-data demo. The dashboard's **# cars** and **layout seed** knobs change the course.

## 6. Perception in the loop — a BEV tracker (N2 + N3)

The ego doesn't get the obstacles' true positions for free. It **observes** them
through a noisy sensor and must *track* them. We reuse N3's tracker design — a
**per-object Kalman filter** (N2's constant-velocity model `[x, y, vx, vy]`) plus
**Hungarian association** with distance gating and a **tentative→confirmed→
coasting→dead** lifecycle — but in the BEV plane instead of the image plane.

When the sensor drops out, matched detections disappear and confirmed tracks
**coast** on their Kalman prediction (exactly the behavior from N2), growing more
uncertain until measurements return.

---

### The per-object filter (N2), in the BEV plane

Each track runs a **constant-velocity Kalman filter** with state $\mathbf{x} = [x, y, v_x, v_y]^\top$, predicted with a constant-velocity model and corrected by position-only measurements $\mathbf{z} = [x, y]^\top$:

$$F = \begin{bmatrix} 1 & 0 & \Delta t & 0 \\ 0 & 1 & 0 & \Delta t \\ 0 & 0 & 1 & 0 \\ 0 & 0 & 0 & 1 \end{bmatrix}, \qquad H = \begin{bmatrix} 1 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 \end{bmatrix}$$

**Predict:** $\quad \mathbf{x}^- = F\mathbf{x}, \qquad P^- = FPF^\top + Q$

**Update:** $\quad \mathbf{y} = \mathbf{z} - H\mathbf{x}^-, \quad S = HP^-H^\top + R, \quad K = P^-H^\top S^{-1}$

$$\mathbf{x}^+ = \mathbf{x}^- + K\mathbf{y}, \qquad P^+ = (I - KH)\,P^-$$

with process noise $Q = \operatorname{diag}(0.5, 0.5, 2, 2)$ and measurement noise $R = \operatorname{diag}(1, 1)$. **Coasting** is just running `predict()` with no `update()`: $P$ grows every step, so the uncertainty ellipse visibly inflates while the sensor is out.

### Association in metric BEV

Just as in N3, we track **points in meters** (not image boxes), so the cost is plain Euclidean distance between each track's predicted position $\hat{\mathbf{p}}_i$ and each detection $\mathbf{z}_j$:

$$C_{ij} = \big\lVert \hat{\mathbf{p}}_i - \mathbf{z}_j \big\rVert_2$$

The Hungarian algorithm picks the minimum-cost assignment, and a **gate** of $4$ m rejects any matched pair farther apart than that (a physically implausible jump), returning both to the unmatched pools — the same distance gating as N3, in metric units.

In [ ]:
class BevTrack:
    """One tracked vehicle: constant-velocity Kalman filter (N2) + lifecycle (N3)."""
    _next = 1
    def __init__(self, z, dt, q_scale=1.0, r_scale=1.0):
        self.id = BevTrack._next; BevTrack._next += 1
        rng = np.random.RandomState(self.id * 7 + 13)
        self.color = tuple(rng.uniform(0.2, 0.9, 3))
        self.F = np.array([[1,0,dt,0],[0,1,0,dt],[0,0,1,0],[0,0,0,1]], float)
        self.H = np.array([[1,0,0,0],[0,1,0,0]], float)
        self.Q = np.diag([0.5, 0.5, 2.0, 2.0]) * q_scale   # process noise (trust in CV model)
        self.R = np.diag([1.0, 1.0]) * r_scale             # measurement noise (trust in detections)
        self.x = np.array([z[0], z[1], 0.0, 0.0])
        self.P = np.diag([2.0, 2.0, 10.0, 10.0])
        self.hits = 1; self.misses = 0; self.state = "tentative"
    def predict(self):
        self.x = self.F @ self.x
        self.P = self.F @ self.P @ self.F.T + self.Q
    def update(self, z):
        y = z - self.H @ self.x
        S = self.H @ self.P @ self.H.T + self.R
        K = self.P @ self.H.T @ np.linalg.inv(S)
        self.x = self.x + K @ y
        self.P = (np.eye(4) - K @ self.H) @ self.P
        self.hits += 1; self.misses = 0
    def pos(self): return self.x[:2]
    def vel(self): return self.x[2:]


class BevTracker:
    """SORT-style tracker in BEV: predict -> Hungarian assign -> update -> birth/death."""
    def __init__(self, dt, gate=4.0, min_hits=2, max_misses=5, q_scale=1.0, r_scale=1.0):
        self.dt = dt; self.gate = gate
        self.min_hits = min_hits; self.max_misses = max_misses
        self.q_scale = q_scale; self.r_scale = r_scale
        self.tracks = []
    def step(self, detections):
        for t in self.tracks:
            t.predict()
        dets = np.array(detections) if len(detections) else np.empty((0, 2))
        matched_t, matched_d = set(), set()
        if self.tracks and len(dets):
            C = np.zeros((len(self.tracks), len(dets)))
            for i, t in enumerate(self.tracks):
                C[i] = np.hypot(dets[:, 0] - t.pos()[0], dets[:, 1] - t.pos()[1])
            for i, j in zip(*linear_sum_assignment(C)):
                if C[i, j] <= self.gate:
                    self.tracks[i].update(dets[j]); matched_t.add(i); matched_d.add(j)
        for i, t in enumerate(self.tracks):
            if i not in matched_t:
                t.misses += 1
                if t.state == "confirmed":
                    t.state = "coasting"
            elif t.hits >= self.min_hits:
                t.state = "confirmed"
        for j in range(len(dets)):
            if j not in matched_d:
                self.tracks.append(BevTrack(dets[j], self.dt, self.q_scale, self.r_scale))
        self.tracks = [t for t in self.tracks if t.misses <= self.max_misses]
        return [t for t in self.tracks if t.state in ("confirmed", "coasting")]

print("BEV tracker ready (N2 Kalman + N3 lifecycle).")

## 7. The new piece — steering around obstacles (potential field)

This is the glue that turns perception + control into a *path around* the cars. We
treat each tracked obstacle as a **repulsor**: any car inside a forward corridor
pushes the ego sideways, away from the side it sits on — stronger the closer and
more head-on it is. Summed over the obstacles and added to the pure-pursuit steer,
the ego **weaves around them and re-centers** on the route. A hard **emergency
brake** backs it up: if a car is dead-ahead within a stopping margin (too close to
swerve), slam the brakes.

Toggle avoidance off and the ego ignores the obstacles entirely — the perfect way
to *see* why it matters (it clips a car).

---

### The repulsion field

For ego heading $\psi$ with forward $\hat{\mathbf t}$ and left $\hat{\mathbf n}$, each tracked obstacle with forward component $f$ and lateral component $\ell$ (relative to the ego) contributes a steering bias only while it is **ahead and in the corridor** ($0 < f < f_{\max}$, $|\ell| < c$):

$$\delta_{\text{avoid}} = \sum_i -\operatorname{sign}(\ell_i)\; g\,\Big(1 - \tfrac{f_i}{f_{\max}}\Big)\Big(1 - \tfrac{|\ell_i|}{c}\Big)$$

with gain $g$ (the dashboard knob), influence $f_{\max} = 16$ m and corridor half-width $c = 5$ m. The total steer is $\delta = \delta_{\text{pursuit}} + \delta_{\text{avoid}}$, clipped to the steering limit. The emergency brake fires when any obstacle is within `brake_dist` straight ahead. `avoidance_steer` and `must_brake` implement these.

## 8. The closed loop — `simulate()`

Now we wire it together into one **pure, deterministic** function: given a
`SimConfig` (all the knobs) it runs the closed loop and returns a log — no plotting.
Each step:

1. **Sense** the static obstacles with noise (skipped during a dropout window).
2. **Track** them (the BEV Kalman tracker) → the ego acts on the *tracked* positions, not god-mode truth.
3. **Steer**: pure-pursuit toward the route **+ the potential-field avoidance bias**.
4. **Brake** if a car is dead-ahead within the stopping margin.
5. **Step** the ego's bicycle model and **check for a clip**.

---

### One discrete-time step

The loop is plain **forward-Euler integration** at a fixed $\Delta t = 0.1$ s (10 Hz): each tick perceives, decides, then advances the true ego state with `bicycle_step`. A clip is logged the first time the true ego comes within $r_{\text{crash}}$ of any obstacle. The function is **seeded and deterministic** — identical `SimConfig` in, identical log out — which is what lets the dashboard render a run and replay it smoothly.

In [ ]:
# --- Static obstacle course: realistic cars (sizes from N3's clustering gates),
# arranged ALONG the real ego path with lateral jitter so the ego must thread them.
# N3's real detections are mostly lateral/moving traffic, so we sample realistic
# sizes/positions and PLACE them on the path (grounded-in-spirit, not literal).
CAR_DIMS = np.array([4.2, 1.8, 1.5])   # [L (X-forward), W (Y-left), H]

def make_obstacle_course(path, n=3, seed=0, jitter=1.6, s_lo=0.25, s_hi=0.85):
    rng = np.random.RandomState(seed)
    s = path_arclength(path)
    obstacles = []
    for f in np.linspace(s_lo, s_hi, n):
        idx = int(np.searchsorted(s, f * s[-1]))
        origin, heading = path_frame(path, idx)
        left = np.array([-np.sin(heading), np.cos(heading)])
        pos = origin + left * rng.uniform(-jitter, jitter)
        obstacles.append({"pos": pos.astype(float), "dims": CAR_DIMS.copy()})
    return obstacles

print("Obstacle course ready:", len(make_obstacle_course(PATH)), "cars on the path")


In [ ]:
# --- Potential-field lateral avoidance -------------------------------------
# Each tracked obstacle inside a forward corridor pushes the ego sideways, AWAY
# from the side it sits on. Closer + more head-on => stronger push. Returns a
# steering BIAS (rad) added to the pure-pursuit steer. Plus a hard emergency brake.
def avoidance_steer(ego, obstacles_xy, gain, influence=16.0, corridor=5.0):
    psi = ego[2]
    ehead = np.array([np.cos(psi),  np.sin(psi)])   # forward unit vector
    eleft = np.array([-np.sin(psi), np.cos(psi)])   # left unit vector
    bias = 0.0
    for p in obstacles_xy:
        rel = np.asarray(p) - ego[:2]
        fwd, lat = rel @ ehead, rel @ eleft
        if 0.0 < fwd < influence and abs(lat) < corridor:
            s = gain * (1.0 - fwd / influence) * (1.0 - abs(lat) / corridor)
            # push AWAY from the obstacle's side: obstacle on left (lat>0) -> negative
            # steer (right); centered (lat==0) -> default to one side to break symmetry
            bias += -(np.sign(lat) if lat != 0 else 1.0) * s
    return float(np.clip(bias, -MAX_STEER, MAX_STEER))

def must_brake(ego, obstacles_xy, brake_dist, half_width=1.4):
    psi = ego[2]
    ehead = np.array([np.cos(psi),  np.sin(psi)])
    eleft = np.array([-np.sin(psi), np.cos(psi)])
    for p in obstacles_xy:
        rel = np.asarray(p) - ego[:2]
        fwd, lat = rel @ ehead, rel @ eleft
        if 0.0 < fwd < brake_dist and abs(lat) < half_width:
            return True
    return False

print("Avoidance helpers ready (potential field + emergency brake).")


In [ ]:
@dataclass
class SimConfig:
    controller: str = "pure_pursuit"   # pure_pursuit | stanley
    lookahead: float = 8.0
    target_speed: float = 8.0
    stanley_k: float = 2.0
    avoidance: bool = True
    avoid_gain: float = 0.9            # potential-field strength
    brake_dist: float = 6.0            # emergency-brake trigger distance (m)
    n_obstacles: int = 3
    obstacle_seed: int = 0
    sensor_noise: float = 0.4
    q_scale: float = 1.0               # Kalman process-noise scale (Q)
    r_scale: float = 1.0               # Kalman measurement-noise scale (R)
    dropout_start: float = -1.0
    dropout_end: float = -1.0
    dt: float = 0.1
    duration: float = 26.0
    seed: int = 0
    crash_dist: float = 2.2
    max_decel: float = 5.0
    max_accel: float = 2.0


def simulate(cfg, path=PATH, movers=None):
    rng = np.random.RandomState(cfg.seed)
    BevTrack._next = 1
    obstacles = make_obstacle_course(path, n=cfg.n_obstacles, seed=cfg.obstacle_seed)
    if movers:
        obstacles = obstacles + list(movers)
    obs_pos = [np.asarray(o["pos"], float).copy() for o in obstacles]   # TRUE positions, advanced each tick
    obs_vel = [np.asarray(o.get("vel", (0.0, 0.0)), float) for o in obstacles]

    p0, h0 = path_frame(path, 0)
    ego = np.array([p0[0], p0[1], h0, cfg.target_speed])
    ego_est = ego.copy()
    tracker = BevTracker(cfg.dt, q_scale=cfg.q_scale, r_scale=cfg.r_scale)

    log = {k: [] for k in ("t", "ego", "ego_est", "tracks", "min_dist", "cte",
                           "braking", "crash", "obstacles", "obstacles_t")}
    crashed = False
    n = int(cfg.duration / cfg.dt)

    for k in range(n):
        t = k * cfg.dt
        in_dropout = cfg.dropout_start >= 0 and cfg.dropout_start <= t < cfg.dropout_end

        # 1. sense the static obstacles (noisy), 2. track them
        dets = [] if in_dropout else [[p[0] + rng.normal(0, cfg.sensor_noise),
                                       p[1] + rng.normal(0, cfg.sensor_noise)] for p in obs_pos]
        tracks = tracker.step(dets)
        track_xy = [tr.pos() for tr in tracks]

        # ego self-estimate (smooth gps / coast under dropout) -- N2 behavior
        if not in_dropout:
            gps = ego[:2] + rng.normal(0, cfg.sensor_noise, 2)
            ego_est[:2] = 0.6 * ego_est[:2] + 0.4 * gps
        else:
            ego_est[0] += ego_est[3] * np.cos(ego_est[2]) * cfg.dt
            ego_est[1] += ego_est[3] * np.sin(ego_est[2]) * cfg.dt
        ego_est[2], ego_est[3] = ego[2], ego[3]

        # 3. steering = pure-pursuit toward path + potential-field avoidance bias
        if cfg.controller == "stanley":
            delta = stanley_steering(ego_est, path, cfg.stanley_k)
        else:
            la, _ = find_lookahead_point(ego_est, path, cfg.lookahead)
            delta = pure_pursuit_steering(ego_est, la)
        if cfg.avoidance and track_xy:
            delta = float(np.clip(delta + avoidance_steer(ego_est, track_xy, cfg.avoid_gain),
                                  -MAX_STEER, MAX_STEER))

        # 4. longitudinal: cruise to target, emergency brake if obstacle dead ahead
        a_cmd = np.clip(cfg.target_speed - ego[3], -cfg.max_decel, cfg.max_accel)
        braking = cfg.avoidance and must_brake(ego_est, track_xy, cfg.brake_dist)
        if braking:
            a_cmd = -cfg.max_decel

        # end of path: stop instead of circling the last waypoint
        if np.hypot(ego[0] - path[-1, 0], ego[1] - path[-1, 1]) < cfg.lookahead:
            delta = 0.0
            a_cmd = np.clip(-ego[3] / cfg.dt, -cfg.max_decel, 0.0)

        # 5. step + crash check (true states)
        ego = bicycle_step(ego, delta, a_cmd, cfg.dt)
        min_dist = min((np.hypot(ego[0] - p[0], ego[1] - p[1]) for p in obs_pos), default=np.inf)
        if min_dist < cfg.crash_dist:
            crashed = True

        log["t"].append(t); log["ego"].append(ego.copy()); log["ego_est"].append(ego_est.copy())
        log["tracks"].append([{"id": tr.id, "pos": tr.pos().copy(), "vel": tr.vel().copy(),
                               "cov": tr.P[:2, :2].copy(), "state": tr.state, "color": tr.color}
                              for tr in tracks])
        log["min_dist"].append(min_dist); log["cte"].append(cross_track_error(ego, path))
        log["braking"].append(braking); log["crash"].append(crashed)
        log["obstacles_t"].append([p.copy() for p in obs_pos])

        obs_pos = [p + v * cfg.dt for p, v in zip(obs_pos, obs_vel)]   # advance TRUE positions (static cars: v=0)

    for kk in ("ego", "ego_est"):
        log[kk] = np.array(log[kk])
    for kk in ("t", "min_dist", "cte"):
        log[kk] = np.array(log[kk])
    log["crashed"] = crashed; log["path"] = path
    log["obstacles"] = obstacles
    return log


_demo = simulate(SimConfig(avoidance=False))
print(f"avoidance OFF -> crashed={_demo['crashed']}, min clearance={_demo['min_dist'].min():.2f} m")


## 9. Looking at one run

Before animating, let's plot a single run statically: the ego-relative BEV at the
end on the left, and the diagnostics (ego speed, |cross-track error|, and min
distance to a car) on the right. Compare avoidance **off** vs **on** below.

In [ ]:
# --- Ego-relative BEV (matches N3's plot_bev_frame convention) -------------
# forward = up, left = left; ego sits a bit ABOVE the bottom so you see behind too.
def world_to_ego(pts_world, ego):
    """World (x-fwd, y-left) -> ego frame (x-fwd, y-left). pts_world: (...,2)."""
    psi = ego[2]; c, s = np.cos(-psi), np.sin(-psi)
    R = np.array([[c, -s], [s, c]])
    return (np.atleast_2d(pts_world) - ego[:2]) @ R.T

def draw_bev_egoframe(ax, log, k, show_trail=True):
    ego = log["ego"][k]
    ax.set_xlim(-25, 25); ax.set_ylim(-12, 52); ax.set_aspect("equal")
    ax.set_facecolor("#eef2ee"); ax.grid(True, alpha=0.2)
    ax.axhline(0, color="gray", lw=0.6, alpha=0.4)          # ego's current axle line
    # reference route (faint), ego frame, plotted (-y, x)
    pe = world_to_ego(log["path"], ego)
    ax.plot(-pe[:, 1], pe[:, 0], color="lightgray", lw=8, alpha=0.6, zorder=0)
    # where the ego has been (trail), transformed into the CURRENT ego frame
    if show_trail and k > 1:
        tr = world_to_ego(log["ego"][:k + 1, :2], ego)
        ax.plot(-tr[:, 1], tr[:, 0], "b-", lw=1.6, alpha=0.55, zorder=3)
    # obstacles = red boxes (per-frame TRUE positions if logged, else static)
    obs_now = log.get("obstacles_t")
    for oi, o in enumerate(log["obstacles"]):
        pos = obs_now[k][oi] if obs_now else o["pos"]
        oe = world_to_ego(np.array([pos]), ego)[0]
        L, W = o["dims"][0], o["dims"][1]
        ax.add_patch(mpatches.Rectangle((-oe[1] - W / 2, oe[0] - L / 2), W, L,
                     facecolor="red", edgecolor="darkred", alpha=0.45, zorder=4))
    # tracked detections + the Kalman velocity ESTIMATE (arrow); near-zero for static cars
    psi = ego[2]; c, s = np.cos(-psi), np.sin(-psi)
    for tr in log["tracks"][k]:
        te = world_to_ego(np.array([tr["pos"]]), ego)[0]
        ax.plot(-te[1], te[0], "o", color=tr["color"], ms=6, zorder=5)
        v = np.asarray(tr["vel"])
        if np.hypot(v[0], v[1]) > 0.5:
            ve = np.array([c * v[0] - s * v[1], s * v[0] + c * v[1]])
            ax.arrow(-te[1], te[0], -ve[1] * 0.7, ve[0] * 0.7, head_width=0.7,
                     length_includes_head=True, color=tr["color"], alpha=0.85, zorder=5)
    # ego: blue triangle at origin, pointing up (forward)
    tri = np.array([[0.0, 2.2], [-1.1, -1.1], [1.1, -1.1]])
    ax.add_patch(mpatches.Polygon(tri, closed=True, facecolor="tab:blue",
                                  edgecolor="black", zorder=6))
    ax.set_xlabel(r"$\leftarrow$ Left      Right $\rightarrow$")
    ax.set_ylabel("Forward (m)")

print("Ego-relative BEV drawing ready (forward-up, ego raised, trail).")


In [ ]:
def plot_run(log, title=""):
    fig, (axm, axd) = plt.subplots(1, 2, figsize=(13, 7), gridspec_kw={"width_ratios": [2, 3]})
    # WORLD overview, forward-up (-y, x): see the entire maneuver at once
    Pw = log["path"]; E = log["ego"]
    axm.plot(-Pw[:, 1], Pw[:, 0], color="lightgray", lw=8, alpha=0.6, zorder=0, label="route")
    axm.plot(-E[:, 1], E[:, 0], "b-", lw=2, zorder=3, label="ego path")
    obs_now = log.get("obstacles_t")
    for oi, o in enumerate(log["obstacles"]):
        L, W = o["dims"][0], o["dims"][1]
        moved = obs_now is not None and np.hypot(*(np.asarray(obs_now[-1][oi]) - np.asarray(obs_now[0][oi]))) > 1.0
        if moved:                                              # swept path + final box
            traj = np.array([f[oi] for f in obs_now])
            axm.plot(-traj[:, 1], traj[:, 0], "--", color="darkred", lw=1.2, alpha=0.6, zorder=3)
            px, py = obs_now[-1][oi]
        else:
            px, py = o["pos"]
        axm.add_patch(mpatches.Rectangle((-py - W / 2, px - L / 2), W, L,
                      facecolor="red", edgecolor="darkred", alpha=0.5, zorder=4))
    axm.plot(-E[0, 1], E[0, 0], "g^", ms=12, zorder=5, label="start")
    if log["crashed"]:
        i = int(np.argmin(log["min_dist"]))
        axm.plot(-E[i, 1], E[i, 0], "x", color="red", ms=20, mew=4, zorder=6, label="clip")
    axm.set_aspect("equal"); axm.grid(alpha=0.3); axm.legend(loc="best", fontsize=8)
    axm.set_xlabel(r"$\leftarrow$ Left      Right $\rightarrow$"); axm.set_ylabel("Forward (m)")
    axm.set_title((title + "  ") + ("CRASH" if log["crashed"] else "threaded"))

    t = log["t"]
    axd.plot(t, log["ego"][:, 3], "b-", label="ego speed (m/s)")
    axd.plot(t, log["min_dist"], "k-", label="min dist to obstacle (m)")
    axd.plot(t, np.abs(log["cte"]), "g-", alpha=0.6, label="|cross-track err| (m)")
    axd.axhline(2.2, color="r", ls=":", lw=0.8, alpha=0.7, label="crash dist")
    br = np.array(log["braking"], bool)
    axd.fill_between(t, 0, 1, where=br, transform=axd.get_xaxis_transform(),
                     color="red", alpha=0.12, label="braking")
    axd.set_xlabel("time (s)"); axd.set_ylim(0, 14); axd.grid(alpha=0.3)
    axd.legend(loc="upper right", fontsize=8); axd.set_title("Diagnostics")
    plt.tight_layout(); plt.show()

plot_run(simulate(SimConfig(avoidance=False)), "Avoidance OFF:")
plot_run(simulate(SimConfig(avoidance=True)),  "Avoidance ON:")


## 10. The BEV animation

Now the same run, animated from above in the **ego's own frame** — forward is up and
you sit at the bottom-center (exactly N3's bird's-eye view). The **red boxes** are
the obstacles, the colored dots are the **tracked** detections the ego steers
around, and the faint gray line is the route. The banner shows speed and flashes
**BRAKING**; the scene flashes red on a clip.

In [ ]:
def animate_run(log, step=2, show_trail=True, figsize=(7.5, 9)):
    idxs = list(range(0, len(log["t"]), step))
    fig, ax = plt.subplots(figsize=figsize)

    def draw(fi):
        ax.clear()
        k = idxs[fi]
        draw_bev_egoframe(ax, log, k, show_trail=show_trail)
        e = log["ego"][k]
        msg = f"t={log['t'][k]:.1f}s  v={e[3]:.1f} m/s" + ("  BRAKING" if log["braking"][k] else "")
        ax.text(0.02, 0.98, msg, transform=ax.transAxes, va="top", fontsize=11,
                fontweight="bold", bbox=dict(boxstyle="round", fc="white", alpha=0.85))
        if log["crash"][k]:
            ax.text(0.5, 0.5, "CRASH", transform=ax.transAxes, ha="center", va="center",
                    fontsize=40, color="red", fontweight="bold", alpha=0.6)

    anim = FuncAnimation(fig, draw, frames=len(idxs), interval=80)
    plt.close(fig)
    return HTML(anim.to_jshtml())

animate_run(simulate(SimConfig(avoidance=True)))


## 11. Your turn — tune the knobs and render

Set the controls, then click **▶ Render run** (or a preset). Each render simulates
the run from scratch and plays it back as a **smooth bird's-eye animation** with a
diagnostics panel underneath. Rendering takes a couple of seconds; playback is smooth.

Things to try:
- *Clip a car*: turn **avoidance** off and watch the ego drive straight into an obstacle.
- *Thread the course*: turn it back on — the potential field weaves it through.
- Push **avoid gain** up for a sharper swerve, or down until it clips.
- Add more **# cars** or change the **layout seed** for a new course.
- *Sensor blackout*: avoidance on, but open a **dropout** window — the ego goes blind
  and can clip a car it can no longer see.

In [ ]:
import ipywidgets as W
from IPython.display import display

# --- driving parameters ---
ctrl   = W.Dropdown(options=["pure_pursuit", "stanley"], value="pure_pursuit", description="controller")
look   = W.FloatSlider(value=8, min=2, max=20, step=1, description="lookahead")
speed  = W.FloatSlider(value=8, min=3, max=14, step=1, description="ego speed")
avoid  = W.Checkbox(value=True, description="avoidance")
trail  = W.Checkbox(value=True, description="show trail")
gain   = W.FloatSlider(value=0.9, min=0, max=1.4, step=0.1, description="avoid gain")
brake  = W.FloatSlider(value=6, min=2, max=12, step=0.5, description="brake dist")
nobs   = W.IntSlider(value=3, min=1, max=6, step=1, description="# cars")
oseed  = W.IntSlider(value=0, min=0, max=9, step=1, description="layout seed")
noise  = W.FloatSlider(value=0.4, min=0, max=3, step=0.1, description="sensor noise")
drop   = W.FloatRangeSlider(value=[0, 0], min=0, max=26, step=0.5, description="dropout")

def _cfg():
    d0, d1 = drop.value
    return SimConfig(controller=ctrl.value, lookahead=look.value, target_speed=speed.value,
                     avoidance=avoid.value, avoid_gain=gain.value, brake_dist=brake.value,
                     n_obstacles=nobs.value, obstacle_seed=oseed.value, sensor_noise=noise.value,
                     dropout_start=(d0 if d1 > d0 else -1.0), dropout_end=(d1 if d1 > d0 else -1.0))

# render-on-demand: simulate once per click and play back a pre-rendered (smooth) animation.
out = W.Output()

def render(_=None):
    with out:
        out.clear_output(wait=True)
        log = simulate(_cfg())
        tag = "CRASH" if log["crashed"] else "threaded the course"
        print(f"{tag}   min clearance = {log['min_dist'].min():.2f} m"
              + ("   (emergency brake fired)" if any(log["braking"]) else ""))
        display(animate_run(log, show_trail=trail.value))
        plot_run(log)

run_btn = W.Button(description="▶ Render run", button_style="success")
run_btn.on_click(render)

def preset(name):
    def _apply(_):
        avoid.value = (name != "clip")
        oseed.value = 0; nobs.value = 3
        drop.value = [5.0, 9.0] if name == "blackout" else [0, 0]
        render()
    return _apply

p_clean = W.Button(description="Clean thread");        p_clean.on_click(preset("clean"))
p_clip  = W.Button(description="No avoidance -> clip"); p_clip.on_click(preset("clip"))
p_black = W.Button(description="Sensor blackout");      p_black.on_click(preset("blackout"))

controls = W.VBox([
    W.HBox([ctrl, look, speed]),
    W.HBox([avoid, trail, gain, brake]),
    W.HBox([nobs, oseed, noise]),
    W.HBox([drop]),
    W.HBox([run_btn, W.Label("set the knobs, then render")]),
    W.HBox([W.Label("Presets:"), p_clean, p_clip, p_black]),
])
display(controls, out)
render()   # initial render so the cell isn't blank


## 12. A moving obstacle: the Kalman filter earns its keep

Until now every car sat still, so the constant-velocity Kalman filter from **N2** had an
easy job — the velocity it estimates just hovered near zero. Here we drop in a **car that
crosses the route**, and the filter's velocity estimate suddenly *matters*: it's what lets
the tracker follow the moving car, draw a sensible velocity arrow, and — crucially —
**coast in the right direction** when the sensor drops out.

This cell exposes two knobs that were hidden inside the tracker:

- **process noise `Q`** — how much the filter trusts its constant-velocity *model*. Too low
  and the track lags behind real motion; too high and it chases measurement noise.
- **measurement noise `R`** — how much it trusts each *detection*. Too low and the track
  jitters with the noise; too high and it smears / lags behind the true car.

Try this: turn **sensor noise** up and watch the raw track jitter, then raise **`R`** to
smooth it (at the cost of lag). Then open a **dropout** window over the moment the car
crosses — with a good velocity estimate the track keeps *moving* through the blackout
instead of freezing in place. The velocity arrow on the track shows what the filter thinks.


In [ ]:
# A car that comes TOWARD the ego and cuts across its lane, tracked live by N2's Kalman.
def make_crossing_car(path, s_frac=0.35, speed=6.0, side=1, offset=6.0):
    """Car spawned ahead of the ego (arclength `s_frac`) and off to one `side`, driving BACK
    toward the ego AND across its lane -- an oncoming car that cuts in front of you."""
    s = path_arclength(path)
    idx = int(np.searchsorted(s, s_frac * s[-1]))
    origin, heading = path_frame(path, idx)
    fwd  = np.array([np.cos(heading), np.sin(heading)])    # along the route (ego's travel dir)
    left = np.array([-np.sin(heading), np.cos(heading)])
    pos = origin + side * offset * left                    # start ahead of the ego, to one side
    direction = -(fwd + side * left)                       # head back toward the ego AND across
    vel = speed * direction / np.hypot(*direction)
    return {"pos": pos.astype(float), "vel": vel.astype(float), "dims": CAR_DIMS.copy()}

# --- mini dashboard: the driving knobs + the two Kalman knobs (Q, R) ---
m_ctrl  = W.Dropdown(options=["pure_pursuit", "stanley"], value="pure_pursuit", description="controller")
m_speed = W.FloatSlider(value=8, min=3, max=14, step=1, description="ego speed")
m_cross = W.FloatSlider(value=6, min=0, max=12, step=0.5, description="car speed")
m_avoid = W.Checkbox(value=True, description="avoidance")
m_gain  = W.FloatSlider(value=0.9, min=0, max=1.4, step=0.1, description="avoid gain")
m_trail = W.Checkbox(value=True, description="show trail")
m_noise = W.FloatSlider(value=0.4, min=0, max=3, step=0.1, description="sensor noise")
m_q     = W.FloatSlider(value=1.0, min=0.1, max=5, step=0.1, description="Q (process)")
m_r     = W.FloatSlider(value=1.0, min=0.1, max=10, step=0.1, description="R (measure)")
m_drop  = W.FloatRangeSlider(value=[0, 0], min=0, max=26, step=0.5, description="dropout")

def _mcfg():
    d0, d1 = m_drop.value
    return SimConfig(controller=m_ctrl.value, target_speed=m_speed.value, avoidance=m_avoid.value,
                     avoid_gain=m_gain.value, sensor_noise=m_noise.value, q_scale=m_q.value,
                     r_scale=m_r.value, n_obstacles=0,        # no static cars: just the crosser
                     dropout_start=(d0 if d1 > d0 else -1.0), dropout_end=(d1 if d1 > d0 else -1.0))

m_out = W.Output()
def m_render(_=None):
    with m_out:
        m_out.clear_output(wait=True)
        car = make_crossing_car(PATH, speed=m_cross.value)
        log = simulate(_mcfg(), movers=[car])
        tag = "CRASH" if log["crashed"] else "threaded"
        print(f"{tag}   min clearance = {log['min_dist'].min():.2f} m   "
              f"(Q={m_q.value:.1f}, R={m_r.value:.1f}, sensor noise={m_noise.value:.1f})")
        display(animate_run(log, show_trail=m_trail.value, figsize=(10, 12)))
        plot_run(log, "Crossing car:")

m_btn = W.Button(description="▶ Render run", button_style="success")
m_btn.on_click(m_render)
display(W.VBox([
    W.HBox([m_ctrl, m_speed, m_cross]),
    W.HBox([m_avoid, m_gain, m_trail]),
    W.HBox([m_noise, m_q, m_r]),
    W.HBox([m_drop]),
    W.HBox([m_btn, W.Label("tune Q / R and the dropout window, then render")]),
]), m_out)
m_render()


## 13. (Optional bonus) Sourcing the obstacles from *real* detections

The obstacles above are realistic but **placed** by us. As a bonus that closes the
loop back to N1/N3, here's how you'd source them from a **real detector**: run YOLO
on a KITTI camera frame, take a vehicle's box, and **back-project** its
ground-contact point to the BEV ground plane (N1's `backproject_to_ground`) — the
same metric frame the course lives in.

This cell is **optional and guarded** — it needs the KITTI images plus the ~237 MB
YOLOv3 weights (the same detector as Workshop 4.2). If they aren't present it
explains what it would do and skips. The course above doesn't depend on it.

In [ ]:
import urllib.request
os.makedirs("models", exist_ok=True)

# Auto-fetch YOLOv3 (the same detector as Workshop 4.2) if not already in models/
_yolo_files = {
    "models/yolov3.cfg":     "https://raw.githubusercontent.com/pjreddie/darknet/master/cfg/yolov3.cfg",
    "models/yolov3.weights": "https://pjreddie.com/media/files/yolov3.weights",   # ~237 MB, one-time
}
import shutil
for _path, _url in _yolo_files.items():
    if not os.path.exists(_path):
        try:
            print(f"Downloading {_path}" + (" (~237 MB, one-time)..." if _path.endswith(".weights") else "..."))
            _req = urllib.request.Request(_url, headers={"User-Agent": "Mozilla/5.0"})  # pjreddie 403s the default UA
            with urllib.request.urlopen(_req, timeout=120) as _r, open(_path, "wb") as _f:
                shutil.copyfileobj(_r, _f)
        except Exception as _e:
            print(f"  could not download {_path}: {_e}")
            if os.path.exists(_path):
                os.remove(_path)   # drop any partial/empty file so the presence check stays honest

KITTI_PRESENT = os.path.isdir(os.path.join("kitti_data", "2011_09_26",
                               "2011_09_26_drive_0005_sync", "image_02", "data"))
WEIGHTS_PRESENT = os.path.exists("models/yolov3.weights") and os.path.exists("models/yolov3.cfg")

if not (KITTI_PRESENT and WEIGHTS_PRESENT):
    print("Bonus skipped:")
    if not KITTI_PRESENT:
        print("  - KITTI drive not found — run N1 first (it downloads the drive into kitti_data/).")
    if not WEIGHTS_PRESENT:
        print("  - YOLOv3 weights/cfg missing from models/ — the auto-download above may have failed; add them manually and re-run.")
else:
    import cv2, pykitti
    print("KITTI + YOLO present - back-projecting a real detection into the BEV scene.")
    # 1) load a frame + calibration (same as N1)
    data = pykitti.raw("kitti_data", "2011_09_26", "0005")
    img = np.array(data.get_cam2(0))
    P2 = np.array(data.calib.P_rect_20)
    R0 = np.array(data.calib.R_rect_00)
    Tr = np.array(data.calib.T_cam0_velo)
    R0e = np.eye(4); R0e[:3, :3] = R0[:3, :3] if R0.shape == (3, 3) else R0[:3, :3]
    M = P2 @ (R0 if R0.shape == (4, 4) else R0e) @ Tr

    def backproject_to_ground(uv, M, z_ground=-1.6):
        u, v = float(uv[0]), float(uv[1])
        a = M[0, 2]*z_ground + M[0, 3]; b = M[1, 2]*z_ground + M[1, 3]; c = M[2, 2]*z_ground + M[2, 3]
        A = np.array([[M[0, 0]-u*M[2, 0], M[0, 1]-u*M[2, 1]],
                      [M[1, 0]-v*M[2, 0], M[1, 1]-v*M[2, 1]]])
        rhs = np.array([u*c-a, v*c-b])
        XY = np.linalg.solve(A, rhs)
        return XY   # [X_forward, Y_left] in velodyne/BEV frame

    # 2) detect vehicles with YOLO (same detector as Workshop 4.2)
    net = cv2.dnn.readNet("models/yolov3.weights", "models/yolov3.cfg")
    ln = net.getLayerNames(); out_layers = [ln[i-1] for i in net.getUnconnectedOutLayers()]
    H, Wd = img.shape[:2]
    blob = cv2.dnn.blobFromImage(img, 1/255.0, (416, 416), swapRB=True, crop=False)
    net.setInput(blob)
    rects, confs = [], []
    for o in net.forward(out_layers):
        for det in o:
            score = float(det[5:].max())
            if score >= 0.5 and int(np.argmax(det[5:])) in (2, 5, 7):   # car / bus / truck
                cx, cy, w, h = det[0]*Wd, det[1]*H, det[2]*Wd, det[3]*H
                rects.append([int(cx - w/2), int(cy - h/2), int(w), int(h)])
                confs.append(score)
    # Non-max suppression: collapse the many overlapping YOLO boxes to ONE per vehicle
    keep = cv2.dnn.NMSBoxes(rects, confs, 0.5, 0.4)
    keep = [int(i) for i in np.array(keep).flatten()] if len(keep) else []

    ground_px, bev_pts = [], []
    for i in keep:
        x, y, w, h = rects[i]
        gx, gy = x + w/2.0, y + h            # bottom-center = where the vehicle meets the road
        XY = backproject_to_ground((gx, gy), M)
        if 0 < XY[0] < 60 and abs(XY[1]) < 15:
            ground_px.append((gx, gy)); bev_pts.append(XY)
    bev = np.array(bev_pts) if bev_pts else np.empty((0, 2))

    fig, (a1, a2) = plt.subplots(1, 2, figsize=(16, 5))
    a1.imshow(img); a1.set_title("KITTI camera frame — YOLO vehicle detections (NMS)"); a1.axis("off")
    for i in keep:
        x, y, w, h = rects[i]
        a1.add_patch(plt.Rectangle((x, y), w, h, fill=False, edgecolor="red", lw=2))
    for idx, (gx, gy) in enumerate(ground_px):
        a1.plot(gx, gy, "yo", ms=7)                                   # ground-contact pixel used
        a1.text(gx, gy + 14, str(idx), color="yellow", fontsize=11, ha="center", fontweight="bold")
    # BEV in the SAME convention as the simulator: forward up, ego bottom-center, (-Y, X)
    if len(bev):
        for idx, (X, Y) in enumerate(bev):
            a2.add_patch(mpatches.Rectangle((-Y - 0.9, X - 2.1), 1.8, 4.2,
                         facecolor="red", edgecolor="darkred", alpha=0.5))
            a2.text(-Y, X + 3.0, str(idx), color="red", fontsize=11, ha="center", fontweight="bold")
    a2.add_patch(mpatches.Polygon(np.array([[0, 2.2], [-1.1, -1.1], [1.1, -1.1]]),
                 closed=True, facecolor="tab:blue", edgecolor="black"))
    _fmax = float(max(25.0, (bev[:, 0].max() + 8) if len(bev) else 25.0))
    a2.set_xlim(-15, 15); a2.set_ylim(-3, _fmax); a2.set_aspect("equal"); a2.grid(alpha=0.3)
    a2.set_xlabel(r"$\leftarrow$ Left      Right $\rightarrow$"); a2.set_ylabel("Forward (m)")
    a2.set_title(f"Real detections in BEV ({len(bev)} vehicles) — forward up, ego at bottom")
    plt.tight_layout(); plt.show()
    for idx, ((gx, gy), (X, Y)) in enumerate(zip(ground_px, bev)):
        print(f"  vehicle {idx}: ground pixel ({gx:.0f},{gy:.0f}) -> BEV  X={X:.1f} m forward, Y={Y:.1f} m left")
    print("These real BEV positions could seed the obstacle course in simulate().")

## Summary

This capstone closed the loop on Workshop 4.3 by fusing every prior notebook into
one interactive BEV simulator on the real KITTI route:

1. **Shared BEV frame (N1)** — a single metric ground plane, anchored on the real
   KITTI drive-0005 ego trajectory.
2. **Control (N4)** — a bicycle model + pure-pursuit/Stanley controller steers the
   ego along that route.
3. **Obstacle course** — realistic cars (N3-sized) placed on the route as **static
   obstacles** the ego has to get around.
4. **Perception & tracking (N2 + N3)** — a per-object constant-velocity Kalman
   tracker turns noisy detections into stable, identified tracks, and coasts through
   sensor dropout.
5. **Decision-making (new)** — a **potential-field avoidance** layer steers the ego
   *around* the obstacles, backed by an emergency brake, producing the
   **thread-vs-clip** outcomes.

### Key takeaways
- **Estimation quality gates safety.** The "sensor blackout" preset clips a car even
  with avoidance on — you can't steer around what you can't perceive. This is why
  N2's estimation and N3's tracking matter, not just the controller.
- **A controller alone isn't autonomy.** N4 keeps the ego on its route beautifully,
  but without the perception → tracking → avoidance chain it drives straight into a
  parked car.
- **Modularity pays off.** Each notebook's component slotted in behind a clean
  interface — the same lesson that lets real autonomy stacks swap detectors,
  trackers, or planners independently.

### Extensions to explore
- Replace the potential field with an explicit **local replanner** (offset path /
  spline around the obstacles), then pure-pursuit the new path.
- Swap the synthetic-fallback route for **A\*** output from Workshop 4.2.
- Add **moving** obstacles (give the placed cars a velocity) and bring back a
  forward-collision brake.
- Source the obstacle course directly from **N3's live detections** (the bonus pipeline).